<a href="https://colab.research.google.com/github/LeonardooAlves/WM9B7-AIDL/blob/main/Week%202/3_CNN_Exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### CNN Exercises for tutorial 2

In this exercise you will be asked to experiment with different CNN setups and hyperparameters, and think about the implications of striving for perfect accuracy on the test set.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch
from torch import nn, optim

In [ ]:
# create directory and download MNIST if not already present
from pathlib import Path
import requests
import pickle
import gzip

DATA_PATH = Path("data")
PATH = DATA_PATH / "mnist"
FILENAME = "mnist.pkl.gz"

with gzip.open((PATH / FILENAME).as_posix(), "rb") as f:
        ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(f, encoding="latin-1")

In [ ]:
# before we can work with them, they need to be converted
# into torch.tensors

x_train, y_train, x_valid, y_valid = map(
    torch.tensor, (x_train, y_train, x_valid, y_valid)
)
train_size, c = x_train.shape


print(x_train, y_train)
print(x_train.shape)
print(y_train.min(), y_train.max())

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]) tensor([5, 0, 4,  ..., 8, 4, 8])
torch.Size([50000, 784])
tensor(0) tensor(9)


# 1. Architectural experiments

Let's try making CNNs with different architertures. Can you get over 90% test accuracy? Try altering the following architectural/modelling choices:

* number of convolutional and dense layers
* convolutional filter sizes, the number of filters at each layer, and the stride length
* type of pooling operation (e.g. ```F.max_pool2d()``` or ```F.avg_pool2d```)
* pooling filter size and stride length
* type of non-linearity (ReLU vs sigmoid vs tanh etc.)
* optimizer (try SGD with momentum perhaps)

The most important thing here is to make sure that the dimensions you choose for adjacent layers mesh together without any problems. You can make a network as complex as you want for the purpose of this - even if you don't have enough computational power to actually train the network, you'll know that the settings are correct if there aren't any errors.

Additionally, you may want to experiment with different combinations of hyperparameters including

* learning rate
* number of epochs
* batch size

Remember that what works well for one architecture may not necessarily give you good results on another. Choosing good hyperparameters mainly comes down to experience, but algorithmic ways such as [random search and grid search](https://machinelearningmastery.com/hyperparameter-optimization-with-random-search-and-grid-search/) do exist.

In [ ]:
# example solution: 3 conv layers + 2 dense, with different filter setups

class CNN(nn.Module):
    def __init__(self):
        super().__init__()





    def forward(self, x):






        return F.log_softmax(x)

def train():
    model.train()
    for i in range(train_size // batch_size):
        start_i = i * batch_size
        end_i = start_i + batch_size
        x_batch = x_train[start_i:end_i]
        y_batch = y_train[start_i:end_i]
        pred = model(x_batch)
        loss = loss_fn(pred, y_batch)

        loss.backward()
        opt.step()
        opt.zero_grad()

        if i % log_interval == 0:
            losses_storage.append(loss.item())

def test():
    predictions = []
    test_accs = []
    test_losses = []
    test_accs = []
    model.eval()
    with torch.no_grad():
        for i in range(test_size // batch_size_test):
            start_i = i * batch_size_test
            end_i = start_i + batch_size_test
            x_batch = x_valid[start_i:end_i]
            y_batch = y_valid[start_i:end_i]

            out = model(x_batch)
            loss = loss_fn(out, y_batch)
            test_losses.append(loss.item())

            pred = out.max(1)[1]
            acc = (sum(pred.eq(y_batch))/len(y_batch))
            test_accs.append(acc.item())
            predictions.append(pred)

    print("Test set average loss: {:.4f}".format(np.mean(test_losses)))
    print("Test set average accuracy: {:.4f}".format(np.mean(test_accs)))

    return np.stack(predictions).ravel()

In [ ]:
# convert the inputs to the correct dimensions


In [ ]:
# define hyperparameters to use as before

num_epochs =
batch_size =
batch_size_test =
lr =
log_interval = 10

# get the model, loss function, and optimiser
model = CNN()
loss_fn =
opt = optim.SGD(model.parameters(), lr = lr)

losses_storage = []

SyntaxError: invalid syntax (<ipython-input-7-6564d7db53d7>, line 3)

In [ ]:
# run the traning loop

test_size = x_valid.shape[0]
predictions = []

for e in range(1, num_epochs + 1):
    train()
    predictions = test()

In [ ]:
plt.plot(losses_storage)

# 2. misclassifications

Once you have a model whose accuracy you're satisfied with, let's try to take a look at where it's making mistakes.

In [ ]:
# find the indices of the validation set
# where the predicted digit isn't equal to the true label



In [ ]:
# visualise some of them using matplotlib

## Q:

Have a look through these misclassified examples. Do they have anything in common? Can we train a model that is capable of correctly predicting these? Should we?

### Hint: spoilers...
> think back to what we said about overfitting and how we'd like training batches to represent the population as much as possible. Do these misclassified examples represent how the general population usually write these digits, or do they just add noise to the model?

## A:

Since the validation set is just another subset of the entire population, they should be treated as such. It's useful to look at the model performance on the validation set as an indication of how well the model will do on unseen data. However, if we try to push for 100% test accuracy, the model will just be overfitted to the test set and become less generalisable.

In short, theoretically there exists at least one set of model parameters which will yield correct predictions even for digits written in unconventional ways. In pushing towards one such set of parameters the model overfits to the noise in the test set and loses out on generalisability as a trade-off.

## Further experiments
1. **Add a third convolutional layer** and observe the effect on accuracy and training time.
2. **Try a different optimiser** such as `optim.Adam` — does it converge faster?
3. **Add Dropout** (`nn.Dropout`) between the fully-connected layers to reduce overfitting.
4. **Experiment with the learning rate** — what happens with `lr=0.1` or `lr=0.001`?
5. **Plot a confusion matrix** to see which digit pairs are most commonly confused.